# Distribuição espacial das estações AlertaRio

Este notebook mostra a localização geográfica das estações AlertaRio usadas no projeto e suas posições na grade de radar reduzida para `128 x 128`. A figura produzida é adequada para inspeção do mapeamento estação-pixel e para inclusão em documentação técnica.

In [ ]:
from html import escape
from pathlib import Path

import folium
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_project_root() -> Path:
    for directory in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (directory / 'data' / 'mapeamento_pixel_estacao_alertario.csv').is_file():
            return directory
    raise FileNotFoundError('Execute o notebook a partir do repositorio ou de notebooks/.')


PROJECT_ROOT = find_project_root()
MAPPING_PATH = PROJECT_ROOT / 'data' / 'mapeamento_pixel_estacao_alertario.csv'
GRID_PATH = PROJECT_ROOT / 'data' / 'sumare_radar_latlon_grid.npz'
FIGURE_PATH = PROJECT_ROOT / 'outputs' / 'analysis' / 'geospatial' / 'mapa_estacoes_alertario.png'
FOLIUM_PATH = PROJECT_ROOT / 'outputs' / 'analysis' / 'geospatial' / 'mapa_estacoes_alertario.html'

print('Projeto:', PROJECT_ROOT)
print('Mapeamento:', MAPPING_PATH)
print('Grade geografica:', GRID_PATH)

In [ ]:
stations = pd.read_csv(MAPPING_PATH)
required_columns = {'station_id', 'nome', 'latitude', 'longitude', 'pixel_i', 'pixel_j'}
missing_columns = required_columns - set(stations.columns)
if missing_columns:
    raise ValueError(f'Colunas ausentes no mapeamento: {sorted(missing_columns)}')

with np.load(GRID_PATH) as grid:
    latitude_grid = grid['lat']
    longitude_grid = grid['lon']

print(f'Estacoes mapeadas: {len(stations)}')
print(f'Grade original: {latitude_grid.shape[0]} x {latitude_grid.shape[1]}')
display(stations.sort_values('station_id').reset_index(drop=True))

## Conversão para a grade usada pelo treinamento

O gerador de targets aplica redimensionamento por vizinho mais próximo. A conversão abaixo reproduz os parâmetros usados na geração dos targets `128 x 128` (`height_orig=656`, `width_orig=654`).

In [ ]:
TARGET_HEIGHT = 128
TARGET_WIDTH = 128
HEIGHT_ORIG = 656
WIDTH_ORIG = 654

stations = stations.copy()
stations['pixel_i_128'] = np.rint(stations['pixel_i'] * TARGET_HEIGHT / HEIGHT_ORIG).astype(int)
stations['pixel_j_128'] = np.rint(stations['pixel_j'] * TARGET_WIDTH / WIDTH_ORIG).astype(int)
stations['pixel_i_128'] = stations['pixel_i_128'].clip(0, TARGET_HEIGHT - 1)
stations['pixel_j_128'] = stations['pixel_j_128'].clip(0, TARGET_WIDTH - 1)

display(stations[['station_id', 'nome', 'latitude', 'longitude', 'pixel_i_128', 'pixel_j_128']].sort_values('station_id'))

## Mapa interativo

O mapa usa a base OpenStreetMap, sem necessidade de chave de API. Os marcadores sempre incluem um popup com identificação e coordenadas. Os rótulos com nomes podem ser ativados pelo controle no canto superior direito ou pela variável `SHOW_STATION_NAMES`.

In [ ]:
SHOW_STATION_NAMES = False
RADAR_LATITUDE = -22.955139
RADAR_LONGITUDE = -43.248278


def grid_boundary(latitude: np.ndarray, longitude: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    rows, columns = latitude.shape
    top = np.arange(columns)
    right = np.arange(1, rows)
    bottom = np.arange(columns - 2, -1, -1)
    left = np.arange(rows - 2, 0, -1)
    row = np.concatenate((np.zeros(columns, dtype=int), right, np.full(columns - 1, rows - 1), left))
    column = np.concatenate((top, np.full(rows - 1, columns - 1), bottom, np.zeros(rows - 2, dtype=int)))
    return latitude[row, column], longitude[row, column]


boundary_latitude, boundary_longitude = grid_boundary(latitude_grid, longitude_grid)
map_center = [stations['latitude'].mean(), stations['longitude'].mean()]
station_map = folium.Map(location=map_center, zoom_start=10, tiles='OpenStreetMap', control_scale=True)

folium.GeoJson(
    {'type': 'Feature', 'geometry': {'type': 'Polygon', 'coordinates': [list(zip(boundary_longitude, boundary_latitude))]}},
    name='Area da grade do Radar Sumare',
    style_function=lambda _: {'color': '#2171b5', 'weight': 2, 'fillColor': '#9ecae1', 'fillOpacity': 0.15},
).add_to(station_map)

folium.Marker(
    [RADAR_LATITUDE, RADAR_LONGITUDE],
    tooltip='Radar Sumare',
    icon=folium.Icon(color='black', icon='tower', prefix='fa'),
).add_to(station_map)

stations_layer = folium.FeatureGroup(name='Estacoes AlertaRio', show=True).add_to(station_map)
names_layer = folium.FeatureGroup(name='Nomes das estacoes', show=SHOW_STATION_NAMES).add_to(station_map)
for station in stations.sort_values('station_id').itertuples():
    popup = (
        f'<b>{escape(str(station.nome))}</b><br>'
        f'ID: {station.station_id}<br>'
        f'Latitude: {station.latitude:.5f}<br>'
        f'Longitude: {station.longitude:.5f}<br>'
        f'Pixel 128 x 128: ({station.pixel_i_128}, {station.pixel_j_128})'
    )
    folium.CircleMarker(
        [station.latitude, station.longitude], radius=5, color='#cb181d', weight=2,
        fill=True, fill_color='#ef3b2c', fill_opacity=0.9, tooltip=str(station.nome), popup=folium.Popup(popup, max_width=280),
    ).add_to(stations_layer)
    folium.Marker(
        [station.latitude, station.longitude],
        icon=folium.DivIcon(html=f'<div style="font-size: 10px; color: #7f0000; white-space: nowrap;">{escape(str(station.nome))}</div>'),
    ).add_to(names_layer)

folium.LayerControl(collapsed=False).add_to(station_map)
FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
station_map.save(FOLIUM_PATH)
print(f'Mapa interativo salvo em: {FOLIUM_PATH}')
station_map

In [ ]:
boundary_latitude, boundary_longitude = grid_boundary(latitude_grid, longitude_grid)
fig, (geographic_axis, pixel_axis) = plt.subplots(1, 2, figsize=(18, 8), constrained_layout=True)

geographic_axis.fill(boundary_longitude, boundary_latitude, color='#9ecae1', alpha=0.35, label='Grade do Radar Sumaré')
geographic_axis.plot(boundary_longitude, boundary_latitude, color='#2171b5', linewidth=1)
geographic_axis.scatter(stations['longitude'], stations['latitude'], color='#cb181d', s=42, zorder=3, label='Estação AlertaRio')
geographic_axis.scatter([RADAR_LONGITUDE], [RADAR_LATITUDE], color='black', marker='*', s=150, zorder=4, label='Radar Sumaré')
for station in stations.itertuples():
    geographic_axis.annotate(station.nome, (station.longitude, station.latitude), xytext=(4, 4), textcoords='offset points', fontsize=7)
geographic_axis.set(title='Estações AlertaRio na área de cobertura do radar', xlabel='Longitude', ylabel='Latitude')
geographic_axis.set_aspect(1 / np.cos(np.deg2rad(RADAR_LATITUDE)))
geographic_axis.grid(alpha=0.25)
geographic_axis.legend(loc='upper right')

pixel_axis.scatter(stations['pixel_j_128'], stations['pixel_i_128'], color='#cb181d', s=42)
for station in stations.itertuples():
    pixel_axis.annotate(station.nome, (station.pixel_j_128, station.pixel_i_128), xytext=(4, 4), textcoords='offset points', fontsize=7)
pixel_axis.set(title='Posições supervisionadas na grade 128 x 128', xlabel='Coluna', ylabel='Linha', xlim=(-1, TARGET_WIDTH), ylim=(TARGET_HEIGHT, -1))
pixel_axis.set_aspect('equal')
pixel_axis.set_xticks(np.arange(0, TARGET_WIDTH + 1, 16))
pixel_axis.set_yticks(np.arange(0, TARGET_HEIGHT + 1, 16))
pixel_axis.grid(alpha=0.35)

FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURE_PATH, dpi=200, bbox_inches='tight')
print(f'Figura salva em: {FIGURE_PATH}')
plt.show()

## Interpretação

Os marcadores vermelhos indicam os pixels supervisionados pela loss mascarada. O modelo produz previsões densas para toda a grade, mas recebe supervisão direta somente nas posições que contêm observações das estações. A concentração espacial de estações deve ser considerada na interpretação das métricas e em qualquer uso operacional dos mapas previstos.